In [24]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
from scipy.optimize import fmin_slsqp as min

import tensorflow
import tensorflow.keras as keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.optimizers as opt
from tensorflow.keras import backend as keras_backend

# import keras
# from keras.models import Sequential
# from keras.layers import Dense
# from keras import layers
# from keras.callbacks import EarlyStopping
# import keras.optimizers as opt
# from keras import backend as keras_backend

import matplotlib.pyplot as plt
import math


#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries, \
                                                     find_last_thurs_date_of_month, \
                                                     prev_workday_if_holiday, find_wkly_expries,\
                                                     date_of_prev_thurs

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data

#import traded options parameters and info
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_strikes, generate_mthly_strikes

#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call_vec, BSM_put_vec

#Import implied volatility processing functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_iv, gen_interpolated_iv

#Import Risk-free interest rate generation functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_ir

#Import Stochatic process simulation functions
from ipynb.fs.full.user_defined_vik_functions import gbm1W_simulation, generate_covariance_from_correlation


In [20]:
# Stock Index of Interest
stock_ident = "BANKNIFTY"

# The scope of this code is to hedge one option 
# and scope will be extended to a portfolio
no_of_assets = 1
cor_mat = [[1]]

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2020: [3]}

#List of holidays
holidays_list = [date(2020, 3, 10)]

#Simulation Parameters
no_of_paths = 5000
#Initialisation of Neural Network hyperparameters for pre-training network
batch_size_divisor = 10
no_of_epochs = 1000 # keep 10 for debugging coding errors
no_of_output_nodes = no_of_assets
batch_size = int(no_of_paths/batch_size_divisor)

#No_of_options to hedge
no_opt = 1


#Fix the hidden nodes - fix a mechanism to select strikes
no_of_hidden_nodes = 5




In [21]:
#Function to price European call options
def pricer_vanilla_by_nn(sim_stock_mat, option_type, opt_strike, dt, \
                         r, vol, nnet_model, batch_size, no_of_epochs, week):

    # Current stock price is at index 0. This extracts stock price at the next time step
    stock_vec = sim_stock_mat[:, 1]
        
    # The following steps extract option value/pay-off at the next week
    if (option_type == "CE" and week != 3):
        option_value = BSM_call_vec(stock_vec, opt_strike, r, vol, dt) * no_opt
    elif (option_type == "PE" and week != 3):
        option_value = BSM_put_vec(stock_vec, opt_strike, r, vol, dt) * no_opt

    elif (option_type == "CE" and week == 3):
        option_value = np.maximum(stock_vec - opt_strike, 0) * no_opt
    else:
        option_value = np.maximum(opt_strike - stock_vec, 0) * no_opt
    
    option_value = np.asarray(option_value)
    option_value = option_value.reshape(-1, 1)
        
    #Creating the training data for the neural network model
    # Neural Network Function using RELU activaion function
    # Build Neural Network model(using Stock prices at next exercise time step and their option value)   
    X_train = stock_vec
    X_train = X_train.reshape(-1, 1)

    Y_train = option_value
    Y_train = np.asarray(Y_train)
    Y_train.reshape(-1, 1)

    nnet_output = nnet_model.fit(X_train, Y_train, epochs=no_of_epochs, batch_size=batch_size, 
                                 verbose=1, validation_split=0.3)

    pfl_weights = np.array(nnet_model.layers[1].get_weights())
            
    return pfl_weights


def pre_train_nn(sim_stock_mat, option_type, \
                 opt_strike, dt, \
                 nnet_model, batch_size, no_of_epochs):

    stock_vec = sim_stock_mat[:, 1]
        
    # The following steps extract option value/pay-off at the next time step
    if (option_type == "CE"):
        option_value = np.maximum(stock_vec - opt_strike, 0) * no_opt
    else:
        option_value = np.maximum(opt_strike - stock_vec, 0) * no_opt
        
    option_value = np.asarray(option_value)
    option_value = option_value.reshape(-1, 1)
        
    #Creating the training data for the neural network model
    # Neural Network Function using RELU activaion function
    # Build Neural Network model(using Stock prices at next exercise time step and their option value)
    X_train = stock_vec
    X_train = X_train.reshape(-1, 1)

    Y_train = option_value
    Y_train = np.asarray(Y_train)
    Y_train.reshape(-1, 1, 1)

    nnet_output = nnet_model.fit(X_train, Y_train, epochs=no_of_epochs, batch_size=batch_size, verbose=0, validation_split=0.3)

    return nnet_model


In [22]:
#########################
#User-defined functions #
#########################

def generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, \
                                       dict_wkly_expiries_each_month, \
                                       dict_ce_wkly_strikes, dict_pe_wkly_strikes, \
                                       dict_wkly_spots, dict_wkly_iv, \
                                       prod_type, prod_moneyness, output_path, \
                                       option_type = "CE", no_of_assets=1, cor_mat=[[1]],\
                                       batch_size_divisor=10, no_of_epochs = 1000):
    ce_pfl_weights = []
    pe_pfl_weights = [] 
    cash_weights = []
    for each_month in mthly_expiries_list:
        K = dict_mthly_strikes[each_month.strftime("%d-%b-%Y")]
        weekly_pfl_weights = []
        ce_dict = {}
        pe_dict = {}
        cash_dict = {}
        for week in range(0, 4):
            s_t = dict_wkly_spots[each_month.strftime("%d-%b-%Y")][week]
            
            if (week == 3):
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = each_month
                #This doesn't matter - as in the last week, BS formula equals to pay-off. Vol and r are just inputs
                vol_list = np.array([dict_wkly_iv[each_month.strftime("%d-%b-%Y")][week]])
                r = np.array([dict_wkly_ir[each_month.strftime("%d-%b-%Y")][week]])
            else:
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week + 1]
                vol_list = np.array([dict_wkly_iv[each_month.strftime("%d-%b-%Y")][week+1]])
                r = np.array([dict_wkly_ir[each_month.strftime("%d-%b-%Y")][week+1]])
         
            sim_stock_mat = gbm1W_simulation(no_of_paths, sim_start_date, sim_end_date, \
                                             no_of_assets, s_t, r, vol_list, cor_mat)
            
            call_strikes_hdge_list_pos = np.sort(np.array(dict_ce_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).tolist()
            call_strikes_hdge_list = [-1 * strike for strike in call_strikes_hdge_list_pos]
            put_strikes_hdge_list = np.sort(np.array(dict_pe_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).tolist()
            no_of_hidden_nodes = len(call_strikes_hdge_list) + len(put_strikes_hdge_list)
            
            call_weights = list(np.ones(len(call_strikes_hdge_list)))
            put_weights = list(-1 * np.ones(len(put_strikes_hdge_list)))
            static_layer_weights = call_weights + put_weights
            static_node_weights = call_strikes_hdge_list + put_strikes_hdge_list
            static_weights = [np.array([static_layer_weights]), np.array(static_node_weights)]

            nnet_model = Sequential()
            hidden_layer = Dense(no_of_hidden_nodes, activation='relu', input_dim=no_of_assets)
            hidden_layer.trainable = False
            nnet_model.add(hidden_layer)
            nnet_model.layers[0].set_weights(static_weights) 
#             output_layer = Dense(no_of_output_nodes, activation='linear', kernel_initializer='normal', use_bias=True, kernel_constraint=keras.constraints.NonNeg())
            
            outer_initializer = tensorflow.keras.initializers.Constant(value=(1/no_of_hidden_nodes))
            output_layer = Dense(no_of_output_nodes, activation='linear', kernel_initializer=outer_initializer, use_bias=True)
            
            output_layer.trainable = True
            nnet_model.add(output_layer)
            nnet_model.compile(optimizer=opt.Adam(lr=0.001), loss='mean_squared_error')
            
            dt = float((each_month - sim_end_date).days) / 365
#             dt = float((sim_end_date - sim_start_date).days) / 365
            batch_size = int(no_of_paths/batch_size_divisor)
            nnet_model = pre_train_nn(sim_stock_mat, option_type, \
                                      K, dt, nnet_model, batch_size, no_of_epochs=100)
        
            #Final Params adjustment for the final neural networks
            keras_backend.set_value(nnet_model.optimizer.lr, 5e-4)

            ######################################################################################
            ### Portfolio weights of shorter term options from neural network of European options ###
            ######################################################################################

            pfl_weights = pricer_vanilla_by_nn(sim_stock_mat, option_type, K, dt, \
                          r, vol_list[0], nnet_model, batch_size, no_of_epochs, week=week)
            
            ce_dates_list = [sim_start_date.strftime("%d-%b-%Y") for i in range(0, len(call_strikes_hdge_list))]
            ce_wt_list = np.array(pfl_weights[0]).reshape(-1)[0:len(call_strikes_hdge_list)]
            ce_dict = {"Date": ce_dates_list, "Strike": call_strikes_hdge_list_pos, "Pfl_weights": ce_wt_list}
            df_ce = pd.DataFrame(ce_dict)
            ce_pfl_weights.append(df_ce)
            
            pe_wt_list = np.array(pfl_weights[0]).reshape(-1)[len(call_strikes_hdge_list):]
            pe_dates_list = [sim_start_date.strftime("%d-%b-%Y") for i in range(0, len(put_strikes_hdge_list))]
            pe_dict = {"Date": pe_dates_list, "Strike":put_strikes_hdge_list, "Pfl_weights": pe_wt_list}
            df_pe = pd.DataFrame(pe_dict)
            pe_pfl_weights.append(df_pe)
            
            cash_date = [sim_start_date.strftime("%d-%b-%Y")]
            cash_weight = np.array(pfl_weights[1][0])
            cash_dict = {"Date":cash_date, "Pfl_weights":cash_weight} 
            df_cash = pd.DataFrame(cash_dict)
            cash_weights.append(df_cash)
            
        df_ce_pfl_weights = pd.concat(ce_pfl_weights, axis=0)
        df_pe_pfl_weights = pd.concat(pe_pfl_weights, axis=0)
        df_cash_pfl_weights = pd.concat(cash_weights, axis=0)
    
        df_ce_pfl_weights.to_csv(output_path  + "B3_" + prod_moneyness + "_" + "CE_Pfl_weights" + ".csv", index = False)
        df_pe_pfl_weights.to_csv(output_path + "B3_" + prod_moneyness + "_" + "PE_Pfl_weights" + ".csv", index = False)
        df_cash_pfl_weights.to_csv(output_path  + "B3_" +  prod_moneyness + "_" + "Cash_Pfl_weights" + ".csv", index = False)
    
    return df_ce_pfl_weights, df_pe_pfl_weights, df_cash_pfl_weights

def output_strikes_hedged(prod_type, dict_mthly_strikes, output_path):
    list_of_dates = []
    list_of_prod_type = []
    list_of_strikes = []
    for each_month in dict_mthly_strikes.keys():
        list_of_dates.append(each_month)
        list_of_strikes.append(dict_mthly_strikes[each_month])
        list_of_prod_type.append(prod_type)
    dict_hedged_prod = {"Date":list_of_dates, "prod_type":list_of_prod_type, "Strike":list_of_strikes}
    df_hedged_prod = pd.DataFrame(dict_hedged_prod) 
    df_hedged_prod.to_csv(output_path + "B3_" + prod_type + "_" + "strikes_hedged.csv", index = False)
    return(df_hedged_prod)


In [23]:

#Find the monthly strikes of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, prod_type_lists=["FUT", "CE", "PE"], stock_ident=stock_ident)

#Load weekly strikes of short term options used for hedging monthly options 
dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots = generate_weekly_strikes(dict_wkly_expiries_each_month, input_data_path, stock_ident = "BANKNIFTY")     

#Load monthly strikes of interest as of required moneyness on the begining of month
dict_mthly_strikes = generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list, stock_ident=stock_ident)

#Load weekly IV for Black-Scholes model input and monte-carlo simulation of stock paths
dict_wkly_iv = generate_weekly_iv(dict_wkly_expiries_each_month, dict_wkly_spots, dict_mthly_strikes, prod_type, output_path=output_data_path)

#Load weekly risk free interest rates from futures
#This function to be changed based on updates on risk-free-interest - now kept it as cosntant
dict_wkly_ir = generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, stock_ident)

#Load Static portfolio weights built by neural network
ce_pfl_wts, pe_pfl_wts, cash_pfl_wts = generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, dict_wkly_expiries_each_month, \
                                                        dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots, \
                                                        dict_wkly_iv, prod_type, prod_moneyness, output_data_path, \
                                                        option_type = prod_type, no_of_assets=no_of_assets, cor_mat=cor_mat,\
                                                        batch_size_divisor=batch_size_divisor, no_of_epochs = no_of_epochs) 
# The strike correspnding to the option that is hedged
df_hedged_prod = output_strikes_hedged(prod_type, dict_mthly_strikes, output_data_path)


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set

7/7 [==============================] - 0s 9ms/step - loss: 611073458176.0000 - val_loss: 587865653248.0000
Epoch 121/1000
7/7 [==============================] - 0s 7ms/step - loss: 610275229696.0000 - val_loss: 587091607552.0000
Epoch 122/1000
7/7 [==============================] - 0s 7ms/step - loss: 609471365120.0000 - val_loss: 586322149376.0000
Epoch 123/1000
7/7 [==============================] - 0s 7ms/step - loss: 608671170560.0000 - val_loss: 585556426752.0000
Epoch 124/1000
7/7 [==============================] - 0s 7ms/step - loss: 607875694592.0000 - val_loss: 584792801280.0000
Epoch 125/1000
7/7 [==============================] - 0s 7ms/step - loss: 607081267200.0000 - val_loss: 584031993856.0000
Epoch 126/1000
7/7 [==============================] - 0s 7ms/step - loss: 606294704128.0000 - val_loss: 583270334464.0000
Epoch 127/1000
7/7 [==============================] - 0s 7ms/step - loss: 605503553536.0000 - val_loss: 582512279552.0000
Epoch 128/1000
7/7 [===================

7/7 [==============================] - 0s 7ms/step - loss: 519063797760.0000 - val_loss: 499625492480.0000
Epoch 255/1000
7/7 [==============================] - 0s 7ms/step - loss: 518468272128.0000 - val_loss: 499056148480.0000
Epoch 256/1000
7/7 [==============================] - 0s 7ms/step - loss: 517866684416.0000 - val_loss: 498493063168.0000
Epoch 257/1000
7/7 [==============================] - 0s 7ms/step - loss: 517285412864.0000 - val_loss: 497923424256.0000
Epoch 258/1000
7/7 [==============================] - 0s 7ms/step - loss: 516688216064.0000 - val_loss: 497358569472.0000
Epoch 259/1000
7/7 [==============================] - 0s 7ms/step - loss: 516095148032.0000 - val_loss: 496796467200.0000
Epoch 260/1000
7/7 [==============================] - 0s 7ms/step - loss: 515515482112.0000 - val_loss: 496228990976.0000
Epoch 261/1000
7/7 [==============================] - 0s 7ms/step - loss: 514919333888.0000 - val_loss: 495667773440.0000
Epoch 262/1000
7/7 [===================

7/7 [==============================] - 0s 7ms/step - loss: 447948324864.0000 - val_loss: 431840231424.0000
Epoch 389/1000
7/7 [==============================] - 0s 8ms/step - loss: 447476367360.0000 - val_loss: 431391080448.0000
Epoch 390/1000
7/7 [==============================] - 0s 7ms/step - loss: 447009718272.0000 - val_loss: 430938980352.0000
Epoch 391/1000
7/7 [==============================] - 0s 7ms/step - loss: 446533795840.0000 - val_loss: 430490583040.0000
Epoch 392/1000
7/7 [==============================] - 0s 7ms/step - loss: 446057316352.0000 - val_loss: 430046576640.0000
Epoch 393/1000
7/7 [==============================] - 0s 7ms/step - loss: 445594828800.0000 - val_loss: 429598277632.0000
Epoch 394/1000
7/7 [==============================] - 0s 7ms/step - loss: 445120544768.0000 - val_loss: 429153976320.0000
Epoch 395/1000
7/7 [==============================] - 0s 7ms/step - loss: 444660350976.0000 - val_loss: 428705021952.0000
Epoch 396/1000
7/7 [===================

7/7 [==============================] - 0s 7ms/step - loss: 390713212928.0000 - val_loss: 377560367104.0000
Epoch 523/1000
7/7 [==============================] - 0s 7ms/step - loss: 390329204736.0000 - val_loss: 377195790336.0000
Epoch 524/1000
7/7 [==============================] - 0s 19ms/step - loss: 389942312960.0000 - val_loss: 376833441792.0000
Epoch 525/1000
7/7 [==============================] - 0s 7ms/step - loss: 389559943168.0000 - val_loss: 376470962176.0000
Epoch 526/1000
7/7 [==============================] - 0s 7ms/step - loss: 389176885248.0000 - val_loss: 376108941312.0000
Epoch 527/1000
7/7 [==============================] - 0s 7ms/step - loss: 388794351616.0000 - val_loss: 375747313664.0000
Epoch 528/1000
7/7 [==============================] - 0s 7ms/step - loss: 388411621376.0000 - val_loss: 375386865664.0000
Epoch 529/1000
7/7 [==============================] - 0s 7ms/step - loss: 388030365696.0000 - val_loss: 375026909184.0000
Epoch 530/1000
7/7 [==================

...

7/7 [==============================] - 0s 7ms/step - loss: 2897252096.0000 - val_loss: 1640688000.0000
Epoch 372/1000
7/7 [==============================] - 0s 7ms/step - loss: 2896184064.0000 - val_loss: 1638841472.0000
Epoch 373/1000
7/7 [==============================] - 0s 7ms/step - loss: 2893879040.0000 - val_loss: 1637505408.0000
Epoch 374/1000
7/7 [==============================] - 0s 19ms/step - loss: 2892497408.0000 - val_loss: 1636059392.0000
Epoch 375/1000
7/7 [==============================] - 0s 7ms/step - loss: 2890729984.0000 - val_loss: 1634607872.0000
Epoch 376/1000
7/7 [==============================] - 0s 7ms/step - loss: 2888609792.0000 - val_loss: 1633513088.0000
Epoch 377/1000
7/7 [==============================] - 0s 7ms/step - loss: 2887574272.0000 - val_loss: 1632056832.0000
Epoch 378/1000
7/7 [==============================] - 0s 7ms/step - loss: 2886200576.0000 - val_loss: 1630343424.0000
Epoch 379/1000
7/7 [==============================] - 0s 7ms/step - lo

7/7 [==============================] - 0s 7ms/step - loss: 2686380800.0000 - val_loss: 1473228288.0000
Epoch 510/1000
7/7 [==============================] - 0s 7ms/step - loss: 2684252928.0000 - val_loss: 1471797504.0000
Epoch 511/1000
7/7 [==============================] - 0s 7ms/step - loss: 2682386432.0000 - val_loss: 1470738944.0000
Epoch 512/1000
7/7 [==============================] - 0s 7ms/step - loss: 2680749056.0000 - val_loss: 1470201472.0000
Epoch 513/1000
7/7 [==============================] - 0s 7ms/step - loss: 2679914496.0000 - val_loss: 1469057280.0000
Epoch 514/1000
7/7 [==============================] - 0s 7ms/step - loss: 2678358528.0000 - val_loss: 1467813248.0000
Epoch 515/1000
7/7 [==============================] - 0s 7ms/step - loss: 2676716544.0000 - val_loss: 1466627328.0000
Epoch 516/1000
7/7 [==============================] - 0s 7ms/step - loss: 2675450624.0000 - val_loss: 1465598208.0000
Epoch 517/1000
7/7 [==============================] - 0s 8ms/step - los

7/7 [==============================] - 0s 7ms/step - loss: 2502123008.0000 - val_loss: 1336051968.0000
Epoch 648/1000
7/7 [==============================] - 0s 7ms/step - loss: 2501384192.0000 - val_loss: 1334724864.0000
Epoch 649/1000
7/7 [==============================] - 0s 7ms/step - loss: 2499875840.0000 - val_loss: 1333533696.0000
Epoch 650/1000
7/7 [==============================] - 0s 7ms/step - loss: 2498578176.0000 - val_loss: 1333283072.0000
Epoch 651/1000
7/7 [==============================] - 0s 7ms/step - loss: 2497354496.0000 - val_loss: 1332152832.0000
Epoch 652/1000
7/7 [==============================] - 0s 8ms/step - loss: 2496005120.0000 - val_loss: 1330996864.0000
Epoch 653/1000
7/7 [==============================] - 0s 7ms/step - loss: 2494687744.0000 - val_loss: 1330214912.0000
Epoch 654/1000
7/7 [==============================] - 0s 7ms/step - loss: 2493421824.0000 - val_loss: 1329539072.0000
Epoch 655/1000
7/7 [==============================] - 0s 7ms/step - los

7/7 [==============================] - 0s 7ms/step - loss: 2342044928.0000 - val_loss: 1220188928.0000
Epoch 786/1000
7/7 [==============================] - 0s 7ms/step - loss: 2341053440.0000 - val_loss: 1219114496.0000
Epoch 787/1000
7/7 [==============================] - 0s 7ms/step - loss: 2339641344.0000 - val_loss: 1218260096.0000
Epoch 788/1000
7/7 [==============================] - 0s 7ms/step - loss: 2338385408.0000 - val_loss: 1217618560.0000
Epoch 789/1000
7/7 [==============================] - 0s 7ms/step - loss: 2337538560.0000 - val_loss: 1217480448.0000
Epoch 790/1000
7/7 [==============================] - 0s 7ms/step - loss: 2336755712.0000 - val_loss: 1216251392.0000
Epoch 791/1000
7/7 [==============================] - 0s 7ms/step - loss: 2335093248.0000 - val_loss: 1215744256.0000
Epoch 792/1000
7/7 [==============================] - 0s 7ms/step - loss: 2334265344.0000 - val_loss: 1214922752.0000
Epoch 793/1000
7/7 [==============================] - 0s 7ms/step - los

7/7 [==============================] - 0s 7ms/step - loss: 2201856256.0000 - val_loss: 1123178496.0000
Epoch 924/1000
7/7 [==============================] - 0s 7ms/step - loss: 2200595712.0000 - val_loss: 1122523648.0000
Epoch 925/1000
7/7 [==============================] - 0s 7ms/step - loss: 2199891712.0000 - val_loss: 1121835136.0000
Epoch 926/1000
7/7 [==============================] - 0s 7ms/step - loss: 2198604800.0000 - val_loss: 1121384320.0000
Epoch 927/1000
7/7 [==============================] - 0s 7ms/step - loss: 2197829376.0000 - val_loss: 1121077632.0000
Epoch 928/1000
7/7 [==============================] - 0s 7ms/step - loss: 2197127936.0000 - val_loss: 1120027008.0000
Epoch 929/1000
7/7 [==============================] - 0s 7ms/step - loss: 2195922432.0000 - val_loss: 1119766912.0000
Epoch 930/1000
7/7 [==============================] - 0s 19ms/step - loss: 2194921728.0000 - val_loss: 1119173376.0000
Epoch 931/1000
7/7 [==============================] - 0s 7ms/step - lo